# Huấn luyện supervised 3 lớp từ một cảm biến siêu âm

Notebook này so sánh nhiều mô hình cho ba nhãn:

- `SAFE`: gồm dữ liệu gốc `SAFE`, `RETREATING` và `EMPTY`. `EMPTY` nghĩa là không có người và không có vật.
- `APPROACHING`: người đang tiến lại gần cảm biến.
- `EMERGENCY`: dữ liệu gốc `DANGER`.

Điểm quan trọng: các dòng trong cùng một recording là chuỗi thời gian liên tục, vì vậy notebook **chia train/test theo `source_file`**, không chia ngẫu nhiên từng dòng.

Mục tiêu đánh giá ưu tiên là `macro F1` và `recall` của `EMERGENCY`. Hard safety rule vẫn độc lập với AI: khoảng cách rất gần không được phép bị mô hình hạ mức cảnh báo.

## 1. Cài đặt và import

Nếu chạy trên máy chưa có thư viện, bỏ comment dòng cài đặt bên dưới. Trên ESP32, notebook này được dùng để train/export model; không nên hiểu rằng file `joblib` có thể chạy trực tiếp trên ESP32.

In [ ]:
# Nếu cần, chạy một lần:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
LABEL_ORDER = ["SAFE", "APPROACHING", "EMERGENCY"]


## 2. Tìm và đọc dataset đã xử lý

In [ ]:
def find_dataset():
    candidates = [
        Path.cwd() / "ai_model/data/single_sensor/processed_3class/dataset_3class.csv",
        Path.cwd().parent / "data/single_sensor/processed_3class/dataset_3class.csv",
        Path.cwd().parent.parent / "ai_model/data/single_sensor/processed_3class/dataset_3class.csv",
        Path("/kaggle/input/single-sensor-dataset/processed_3class/dataset_3class.csv"),
        Path("/kaggle/working/single_sensor_dataset/processed_3class/dataset_3class.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    # Fallback: tìm trong các thư mục Kaggle input.
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            matches = list(root.rglob("dataset_3class.csv"))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        "Không tìm thấy dataset_3class.csv. Hãy đặt dataset vào "
        "ai_model/data/single_sensor/processed_3class/"
    )

data_path = find_dataset()
df = pd.read_csv(data_path)
required = {"source_file", "elapsed_s", "distance_cm", "original_label", "label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Thiếu cột: {sorted(missing)}")

df["elapsed_s"] = pd.to_numeric(df["elapsed_s"], errors="coerce")
df["distance_cm"] = pd.to_numeric(df["distance_cm"], errors="coerce")
df = df.dropna(subset=["source_file", "elapsed_s", "distance_cm", "label"]).copy()
df["label"] = df["label"].astype(str).str.upper()
if set(df["label"].unique()) != set(LABEL_ORDER):
    raise ValueError(f"Nhãn không đúng: {sorted(df['label'].unique())}")

print("Dataset:", data_path)
print("Số dòng:", len(df))
print("Số recording:", df["source_file"].nunique())
print(df["label"].value_counts().reindex(LABEL_ORDER).to_string())


In [ ]:
# Kiểm tra mapping nhãn gốc -> nhãn train.
print(pd.crosstab(df["original_label"], df["label"]).reindex(columns=LABEL_ORDER, fill_value=0).to_string())


## 3. Tạo đặc trưng theo thời gian

Một mẫu đơn chỉ có khoảng cách chưa đủ để phân biệt `SAFE` và `APPROACHING`. Ta thêm vận tốc thay đổi khoảng cách trong cùng recording và các thống kê cửa sổ quá khứ.

Quy ước: vận tốc dương nghĩa là khoảng cách tăng, tức là đối tượng đang rời xa; vận tốc âm nghĩa là đang tiến lại gần.

In [ ]:
def make_features(frame):
    out = frame.sort_values(["source_file", "elapsed_s"]).copy()
    grouped = out.groupby("source_file", sort=False)
    dt = grouped["elapsed_s"].diff()
    dd = grouped["distance_cm"].diff()
    out["approach_speed_cm_s"] = (dd / dt.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    # Loại nhiễu cực đoan ở đạo hàm do mẫu thời gian bất thường; khoảng cách gốc vẫn giữ nguyên.
    out["approach_speed_cm_s"] = out["approach_speed_cm_s"].clip(-500, 500).fillna(0.0)
    out["distance_delta_3"] = grouped["distance_cm"].diff(3).fillna(0.0)
    out["distance_std_5"] = grouped["distance_cm"].transform(
        lambda s: s.rolling(5, min_periods=1).std().fillna(0.0)
    )
    out["distance_mean_5"] = grouped["distance_cm"].transform(
        lambda s: s.rolling(5, min_periods=1).mean()
    )
    out["speed_mean_5"] = grouped["approach_speed_cm_s"].transform(
        lambda s: s.rolling(5, min_periods=1).mean()
    )
    return out

feature_df = make_features(df)
FEATURE_COLUMNS = [
    "distance_cm", "approach_speed_cm_s", "distance_delta_3",
    "distance_std_5", "distance_mean_5", "speed_mean_5",
]
X = feature_df[FEATURE_COLUMNS].astype(float)
y = feature_df["label"]
groups = feature_df["source_file"]
print(feature_df[FEATURE_COLUMNS].describe().round(2).to_string())


## 4. Kiểm tra nhanh dữ liệu

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=feature_df, x="label", order=LABEL_ORDER, ax=axes[0])
axes[0].set_title("Phân bố nhãn")
axes[0].set_xlabel("")

sns.boxplot(data=feature_df, x="label", y="distance_cm", order=LABEL_ORDER, ax=axes[1])
axes[1].set_title("Khoảng cách theo nhãn")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()


## 5. Chia nhóm và định nghĩa các mô hình

`StratifiedGroupKFold` bảo đảm các dòng của một recording không bị rơi vào cả train và validation. Vì mỗi recording hiện chỉ có một nhãn, đây là cách chia bảo thủ hơn so với chia từng dòng.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(cv.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train, g_test = groups.iloc[train_idx], groups.iloc[test_idx]

print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("Train recordings:", g_train.nunique(), "Test recordings:", g_test.nunique())
print("Train labels:", y_train.value_counts().reindex(LABEL_ORDER, fill_value=0).to_dict())
print("Test labels:", y_test.value_counts().reindex(LABEL_ORDER, fill_value=0).to_dict())
print("Test recordings:", sorted(g_test.unique()))

models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    ),
}


## 6. So sánh mô hình trên test set theo recording

In [ ]:
def metrics_row(name, y_true, y_pred):
    report = classification_report(y_true, y_pred, labels=LABEL_ORDER, output_dict=True, zero_division=0)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": report["macro avg"]["f1-score"],
        "emergency_recall": report["EMERGENCY"]["recall"],
        "emergency_precision": report["EMERGENCY"]["precision"],
    }

results = []
predictions = {}
for name, model in models.items():
    fitted = clone(model)
    fitted.fit(X_train, y_train)
    pred = fitted.predict(X_test)
    predictions[name] = pred
    results.append(metrics_row(name, y_test, pred))

results_df = pd.DataFrame(results).sort_values(
    ["macro_f1", "emergency_recall", "balanced_accuracy"], ascending=False
).reset_index(drop=True)
results_df.round(4)


In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = clone(models[best_name])
best_model.fit(X_train, y_train)
best_pred = predictions[best_name]
print("Model tốt nhất theo macro F1:", best_name)
print(classification_report(y_test, best_pred, labels=LABEL_ORDER, zero_division=0))

cm = confusion_matrix(y_test, best_pred, labels=LABEL_ORDER)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER)
plt.xlabel("Dự đoán")
plt.ylabel("Nhãn thật")
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()


## 7. Cross-validation theo recording

Một lần chia holdout chỉ cho một góc nhìn. Cell này tính điểm trung bình qua 5 fold; nếu số recording tăng lên, kết quả này sẽ đáng tin cậy hơn.

In [ ]:
cv_results = []
for name, model in models.items():
    fold_rows = []
    for fold, (tr, va) in enumerate(cv.split(X, y, groups=groups), start=1):
        fitted = clone(model)
        fitted.fit(X.iloc[tr], y.iloc[tr])
        pred = fitted.predict(X.iloc[va])
        row = metrics_row(name, y.iloc[va], pred)
        row["fold"] = fold
        fold_rows.append(row)
    cv_results.extend(fold_rows)

cv_results_df = pd.DataFrame(cv_results)
cv_summary = (cv_results_df.groupby("model")[["accuracy", "balanced_accuracy", "macro_f1", "emergency_recall", "emergency_precision"]]
              .agg(["mean", "std"]).sort_values(("macro_f1", "mean"), ascending=False))
cv_summary.round(4)


## 8. Huấn luyện model cuối và lưu artifact

Model được train lại trên toàn bộ dataset sau khi chọn kiến trúc. Artifact chứa feature order, mapping nhãn và ngưỡng bảo vệ để gateway dùng nhất quán.

In [ ]:
# Chọn theo điểm CV macro F1 trung bình; nếu hòa, ưu tiên recall EMERGENCY.
model_order = (cv_results_df.groupby("model")[["macro_f1", "emergency_recall", "balanced_accuracy"]]
               .mean().sort_values(["macro_f1", "emergency_recall", "balanced_accuracy"], ascending=False))
final_name = model_order.index[0]
final_model = clone(models[final_name])
final_model.fit(X, y)

artifact_dir = Path.cwd() / "ai_model/artifacts"
if not artifact_dir.exists():
    artifact_dir = Path.cwd().parent / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

artifact = {
    "model": final_model,
    "model_name": final_name,
    "feature_columns": FEATURE_COLUMNS,
    "label_order": LABEL_ORDER,
    "label_mapping": {"SAFE": 0, "APPROACHING": 1, "EMERGENCY": 2},
    "thresholds_cm": {"emergency_hard": 30.0, "warning": 60.0},
    "training_rows": int(len(X)),
    "training_recordings": int(groups.nunique()),
    "original_label_mapping": {
        "DANGER": "EMERGENCY",
        "APPROACHING": "APPROACHING",
        "SAFE": "SAFE",
        "RETREATING": "SAFE",
        "EMPTY": "SAFE",
    },
}
model_path = artifact_dir / "single_sensor_3class_best.joblib"
metrics_path = artifact_dir / "single_sensor_3class_metrics.json"
joblib.dump(artifact, model_path)
metrics_payload = {
    "selected_model": final_name,
    "feature_columns": FEATURE_COLUMNS,
    "holdout_results": results_df.to_dict(orient="records"),
    "cross_validation_summary": json.loads(cv_summary.round(6).to_json()),
    "labels": LABEL_ORDER,
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
print("Đã lưu model:", model_path)
print("Đã lưu metrics:", metrics_path)


## 9. Hàm dự đoán dùng chung cho gateway

Trong hệ thống thật, gateway cần duy trì khoảng cách trước đó và một buffer ngắn để tạo đúng feature. Phần hard rule được áp dụng trước dự đoán AI.

In [ ]:
def predict_measurement(distance_cm, previous_distance_cm=None, dt_s=0.18):
    distance_cm = float(distance_cm)
    previous = distance_cm if previous_distance_cm is None else float(previous_distance_cm)
    speed = (distance_cm - previous) / max(float(dt_s), 1e-6)
    row = pd.DataFrame([{
        "distance_cm": distance_cm,
        "approach_speed_cm_s": np.clip(speed, -500, 500),
        # Với dự đoán online tối giản, dùng giá trị hiện tại làm fallback cho cửa sổ chưa đầy.
        "distance_delta_3": distance_cm - previous,
        "distance_std_5": 0.0,
        "distance_mean_5": distance_cm,
        "speed_mean_5": np.clip(speed, -500, 500),
    }])[FEATURE_COLUMNS]
    ai_label = final_model.predict(row)[0]
    # Không cho AI hạ emergency do ngưỡng cứng.
    if distance_cm <= 30.0:
        return {"label": "EMERGENCY", "emergency_stop": True, "ai_label": ai_label, "reason": "hard_distance_rule"}
    return {"label": ai_label, "emergency_stop": False, "ai_label": ai_label, "reason": "ai_prediction"}

for distance, previous in [(25, 28), (100, 120), (160, 160)]:
    print(distance, previous, "->", predict_measurement(distance, previous))


## 10. Tiêu chí trước khi triển khai

Không nên triển khai chỉ dựa trên accuracy. Cần kiểm tra riêng recall của `EMERGENCY`, false negative trong confusion matrix, dữ liệu ở điều kiện ánh sáng/nhiệt độ/vật liệu khác nhau và dữ liệu ghi trực tiếp từ đúng cảm biến trên ESP32. Khi chạy thực tế, nếu cảm biến lỗi hoặc mất mẫu, hệ thống nên chuyển sang trạng thái an toàn thay vì coi đó là `SAFE`.